In [1]:
# clone and install
!nvidia-smi
!git clone https://github.com/rubinanoor/small-model-alignment-starter.git /kaggle/working/repo
%cd /kaggle/working/repo
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest peft trl
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print(torch.__version__, transformers.__version__, datasets.__version__, torch.cuda.device_count())
!PYTHONPATH=. python -m pytest -q tests

Thu Oct 15 17:42:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# build the AlpacaEval subset
import json, random, os
from huggingface_hub import hf_hub_download

path = hf_hub_download("tatsu-lab/alpaca_eval", "alpaca_eval.json", repo_type="dataset")
data = json.load(open(path))
print(len(data), data[0].keys())          # expect 805 rows with "instruction"
idx = random.Random(42).sample(range(len(data)), 300)
items = [{"id": f"alpaca-{i}", "instruction": data[i]["instruction"], "dataset": "alpaca_eval"} for i in idx]
os.makedirs("eval_sets", exist_ok=True)
json.dump(items, open("eval_sets/alpaca_300_seed42.json", "w"), indent=1)
print(len(items), items[0]["instruction"][:80])

alpaca_eval.json: 0.00B [00:00, ?B/s]

805 dict_keys(['dataset', 'instruction', 'output', 'generator'])
300 Create a short, concise summary of the paper based on its abstract.

Few-shot le


In [3]:
#tiny sanity check, then the baseline
!mkdir -p outputs/gen outputs/eval
!PYTHONPATH=. python scripts/generate.py Qwen/Qwen2.5-0.5B qwen2.5-0.5b-base eval_sets/alpaca_300_seed42.json outputs/gen/base.json

tokenizer_config.json: 7.23kB [00:00, 21.7MB/s]
vocab.json: 2.78MB [00:00, 73.8MB/s]
merges.txt: 1.67MB [00:00, 110MB/s]
tokenizer.json: 7.03MB [00:00, 185MB/s]
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 4.39MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 988M/988M [00:04<00:00, 224MB/s]
generation_config.json: 100%|██████████████████| 138/138 [00:00<00:00, 1.14MB/s]


In [4]:
#SFT (about 1.5 h)
!PYTHONPATH=. python scripts/train_sft.py --out outputs/sft

`torch_dtype` is deprecated! Use `dtype` instead!
README.md: 3.90kB [00:00, 13.6MB/s]
data/train_sft-00000-of-00003-a3ecf92756(…): 100%|█| 244M/244M [00:02<00:00, 115
data/train_sft-00001-of-00003-0a1804bcb6(…): 100%|█| 244M/244M [00:02<00:00, 114
data/train_sft-00002-of-00003-ee46ed25cf(…): 100%|█| 244M/244M [00:01<00:00, 162
data/test_sft-00000-of-00001-f7dfac4afe5(…): 100%|█| 81.2M/81.2M [00:01<00:00, 4
data/train_gen-00000-of-00003-a6c9fb894b(…): 100%|█| 244M/244M [00:01<00:00, 176
data/train_gen-00001-of-00003-d6a0402e41(…): 100%|█| 243M/243M [00:01<00:00, 170
data/train_gen-00002-of-00003-c0db75b92a(…): 100%|█| 243M/243M [00:01<00:00, 143
data/test_gen-00000-of-00001-3d4cd830914(…): 100%|█| 80.4M/80.4M [00:01<00:00, 4
Generating train_sft split: 100%|█| 207865/207865 [00:03<00:00, 52649.15 example
Generating test_sft split: 100%|█| 23110/23110 [00:00<00:00, 55318.92 examples/s
Generating train_gen split: 100%|█| 256032/256032 [00:03<00:00, 67920.85 example
Generating test_gen spl

In [5]:
#zip and save
!cd outputs && zip -r /kaggle/working/sft_run_outputs.zip sft gen

  adding: sft/ (stored 0%)
  adding: sft/final/ (stored 0%)
  adding: sft/final/generation_config.json (deflated 29%)
  adding: sft/final/merges.txt (deflated 57%)
  adding: sft/final/tokenizer.json (deflated 81%)
  adding: sft/final/config.json (deflated 69%)
  adding: sft/final/chat_template.jinja (deflated 71%)
  adding: sft/final/special_tokens_map.json (deflated 69%)
  adding: sft/final/model.safetensors (deflated 7%)
  adding: sft/final/vocab.json (deflated 61%)
  adding: sft/final/tokenizer_config.json (deflated 89%)
  adding: sft/final/added_tokens.json (deflated 67%)
  adding: sft/best/ (stored 0%)
  adding: sft/best/generation_config.json (deflated 29%)
  adding: sft/best/merges.txt (deflated 57%)
  adding: sft/best/tokenizer.json (deflated 81%)
  adding: sft/best/config.json (deflated 69%)
  adding: sft/best/chat_template.jinja (deflated 71%)
  adding: sft/best/special_tokens_map.json (deflated 69%)
  adding: sft/best/model.safetensors (deflated 7%)
  adding: sft/best/vocab.